In [1]:
import json
import os
import sys
import re
import unicodedata
import pandas as pd

In [2]:
# Readin
df = pd.read_csv('MedSynth_huggingface_final.csv')
df

,Note,Dialogue,ICD10,ICD10_desc
0,**1. Subjective:**\n\n **Chief Complaint (CC...,[doctor]: Hello! It’s good to see you today. H...,M25562,PAIN IN LEFT KNEE
1,**1. Subjective:**\n\n - **Chief Complaint (...,"[doctor] Hi there, how are you today?\n\n[pati...",M25562,PAIN IN LEFT KNEE
2,**1. Subjective:**\n\n**Chief Complaint (CC):*...,"[doctor] Good morning, how are you doing today...",M25562,PAIN IN LEFT KNEE
3,**1. Subjective:**\n\n**Chief Complaint (CC):*...,[doctor] Good morning! How are you feeling tod...,M25562,PAIN IN LEFT KNEE
4,#####\n**1. Subjective:**\n\n**Chief Complaint...,"[doctor]: Hello Mr. Doe, how are you doing tod...",M25562,PAIN IN LEFT KNEE
...,...,...,...,...
10235,#####\n**1. Subjective:**\n \n**Chief Compla...,[doctor]: Good morning. How are you doing toda...,B3781,CANDIDAL ESOPHAGITIS
10236,### Gastroenterologist Medical Note\n\n#### 1....,"**Doctor:** Hi there, how are you doing today?...",B3781,CANDIDAL ESOPHAGITIS
10237,**1. Subjective:**\n\n**Chief Complaint (CC):*...,"[doctor]: Hi Mr. Harris, how are you doing tod...",B3781,CANDIDAL ESOPHAGITIS
10238,#####\n**1. Subjective:**\n**Chief Complaint (...,"[doctor]: Good morning, Ms. Lee. How are you d...",B3781,CANDIDAL ESOPHAGITIS


In [3]:
# Harmonizing UTF characters
def clean_string(s):
    if not isinstance(s, str):
        return s

    s = unicodedata.normalize("NFKC", s)
    s = re.sub(r"[\u200b\u200c\u200d\ufeff]", "", s)
    s = re.sub(r"[\x00-\x1F\x7F]", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

df[["Note", "Dialogue"]] = df[["Note", "Dialogue"]].map(clean_string)

In [4]:
df

,Note,Dialogue,ICD10,ICD10_desc
0,**1. Subjective:** **Chief Complaint (CC):** -...,[doctor]: Hello! It’s good to see you today. H...,M25562,PAIN IN LEFT KNEE
1,**1. Subjective:** - **Chief Complaint (CC):**...,"[doctor] Hi there, how are you today?[patient]...",M25562,PAIN IN LEFT KNEE
2,**1. Subjective:****Chief Complaint (CC):**Sev...,"[doctor] Good morning, how are you doing today...",M25562,PAIN IN LEFT KNEE
3,**1. Subjective:****Chief Complaint (CC):** Mo...,[doctor] Good morning! How are you feeling tod...,M25562,PAIN IN LEFT KNEE
4,#####**1. Subjective:****Chief Complaint (CC):...,"[doctor]: Hello Mr. Doe, how are you doing tod...",M25562,PAIN IN LEFT KNEE
...,...,...,...,...
10235,#####**1. Subjective:** **Chief Complaint (CC)...,[doctor]: Good morning. How are you doing toda...,B3781,CANDIDAL ESOPHAGITIS
10236,### Gastroenterologist Medical Note#### 1. Sub...,"**Doctor:** Hi there, how are you doing today?...",B3781,CANDIDAL ESOPHAGITIS
10237,**1. Subjective:****Chief Complaint (CC):**Dif...,"[doctor]: Hi Mr. Harris, how are you doing tod...",B3781,CANDIDAL ESOPHAGITIS
10238,#####**1. Subjective:****Chief Complaint (CC):...,"[doctor]: Good morning, Ms. Lee. How are you d...",B3781,CANDIDAL ESOPHAGITIS


In [5]:
# Erasing leading styling characters

s = df["Note"].astype("string")
df["Note"] = s.apply(lambda x: x[x.find("**"):] if isinstance(x, str) and "**" in x else x)

In [6]:
df

,Note,Dialogue,ICD10,ICD10_desc
0,**1. Subjective:** **Chief Complaint (CC):** -...,[doctor]: Hello! It’s good to see you today. H...,M25562,PAIN IN LEFT KNEE
1,**1. Subjective:** - **Chief Complaint (CC):**...,"[doctor] Hi there, how are you today?[patient]...",M25562,PAIN IN LEFT KNEE
2,**1. Subjective:****Chief Complaint (CC):**Sev...,"[doctor] Good morning, how are you doing today...",M25562,PAIN IN LEFT KNEE
3,**1. Subjective:****Chief Complaint (CC):** Mo...,[doctor] Good morning! How are you feeling tod...,M25562,PAIN IN LEFT KNEE
4,**1. Subjective:****Chief Complaint (CC):** Mo...,"[doctor]: Hello Mr. Doe, how are you doing tod...",M25562,PAIN IN LEFT KNEE
...,...,...,...,...
10235,**1. Subjective:** **Chief Complaint (CC):**Di...,[doctor]: Good morning. How are you doing toda...,B3781,CANDIDAL ESOPHAGITIS
10236,**Chief Complaint (CC):** Difficulty swallowin...,"**Doctor:** Hi there, how are you doing today?...",B3781,CANDIDAL ESOPHAGITIS
10237,**1. Subjective:****Chief Complaint (CC):**Dif...,"[doctor]: Hi Mr. Harris, how are you doing tod...",B3781,CANDIDAL ESOPHAGITIS
10238,**1. Subjective:****Chief Complaint (CC):** Se...,"[doctor]: Good morning, Ms. Lee. How are you d...",B3781,CANDIDAL ESOPHAGITIS


In [7]:
# Drop NA (and one abnormative) values

df.isna().sum()
df = df.drop([10236])
df = df.dropna()
df = df.sort_values(["ICD10", "Note"])
df.reset_index(inplace=True)
df = df.drop(["index"], axis=1)
df

,Note,Dialogue,ICD10,ICD10_desc
0,**1. Subjective:****Chief Complaint (CC):** Se...,[doctor]: Good morning. How are you feeling to...,A047,ENTEROCOLITIS DUE TO CLOSTRIDIUM DIFFICILE
1,**1. Subjective:****Chief Complaint (CC):**Fre...,"[doctor]: Good morning, Mrs. Doe. How are you ...",A047,ENTEROCOLITIS DUE TO CLOSTRIDIUM DIFFICILE
2,**1. Subjective:****Chief Complaint (CC):**Mod...,"[doctor]: Hi Mr. Lee, how are you doing today?...",A047,ENTEROCOLITIS DUE TO CLOSTRIDIUM DIFFICILE
3,**1. Subjective:****Chief Complaint (CC):**Wat...,"[doctor]: Good morning, how are you today?[pat...",A047,ENTEROCOLITIS DUE TO CLOSTRIDIUM DIFFICILE
4,**Subjective:****Chief Complaint (CC)**Severe ...,"[doctor]: Hi there, how are you doing today?[p...",A047,ENTEROCOLITIS DUE TO CLOSTRIDIUM DIFFICILE
...,...,...,...,...
10232,**1. Subjective:****Chief Complaint (CC):**Fol...,[doctor]: Good morning! It's nice to see you a...,Z9981,DEPENDENCE ON SUPPLEMENTAL OXYGEN
10233,**1. Subjective:****Chief Complaint (CC):**Per...,"```markdown[doctor] Hello, how are you doing t...",Z9981,DEPENDENCE ON SUPPLEMENTAL OXYGEN
10234,**1. Subjective:****Chief Complaint (CC):**Sev...,[doctor] Good morning. How are you feeling tod...,Z9981,DEPENDENCE ON SUPPLEMENTAL OXYGEN
10235,**1. Subjective:****Chief Complaint (CC):**Sev...,"[doctor] Hi Mr. Lee, how are you doing today?[...",Z9981,DEPENDENCE ON SUPPLEMENTAL OXYGEN


In [8]:
# Determining underrepresented ICD codes

counts = df['ICD10'].value_counts()
filtered = counts[counts < 5]

print(filtered)

ICD10
B3781    4
O621     4
Z793     4
Name: count, dtype: int64


In [9]:
# Deleting rows with underrepresented ICD codes

df = df[df['ICD10'].map(df['ICD10'].value_counts()) >= 5]
df

,Note,Dialogue,ICD10,ICD10_desc
0,**1. Subjective:****Chief Complaint (CC):** Se...,[doctor]: Good morning. How are you feeling to...,A047,ENTEROCOLITIS DUE TO CLOSTRIDIUM DIFFICILE
1,**1. Subjective:****Chief Complaint (CC):**Fre...,"[doctor]: Good morning, Mrs. Doe. How are you ...",A047,ENTEROCOLITIS DUE TO CLOSTRIDIUM DIFFICILE
2,**1. Subjective:****Chief Complaint (CC):**Mod...,"[doctor]: Hi Mr. Lee, how are you doing today?...",A047,ENTEROCOLITIS DUE TO CLOSTRIDIUM DIFFICILE
3,**1. Subjective:****Chief Complaint (CC):**Wat...,"[doctor]: Good morning, how are you today?[pat...",A047,ENTEROCOLITIS DUE TO CLOSTRIDIUM DIFFICILE
4,**Subjective:****Chief Complaint (CC)**Severe ...,"[doctor]: Hi there, how are you doing today?[p...",A047,ENTEROCOLITIS DUE TO CLOSTRIDIUM DIFFICILE
...,...,...,...,...
10232,**1. Subjective:****Chief Complaint (CC):**Fol...,[doctor]: Good morning! It's nice to see you a...,Z9981,DEPENDENCE ON SUPPLEMENTAL OXYGEN
10233,**1. Subjective:****Chief Complaint (CC):**Per...,"```markdown[doctor] Hello, how are you doing t...",Z9981,DEPENDENCE ON SUPPLEMENTAL OXYGEN
10234,**1. Subjective:****Chief Complaint (CC):**Sev...,[doctor] Good morning. How are you feeling tod...,Z9981,DEPENDENCE ON SUPPLEMENTAL OXYGEN
10235,**1. Subjective:****Chief Complaint (CC):**Sev...,"[doctor] Hi Mr. Lee, how are you doing today?[...",Z9981,DEPENDENCE ON SUPPLEMENTAL OXYGEN


In [10]:
# Deleting unneccessary columns
df = df.drop(["Note", "ICD10_desc"], axis=1)
df

,Dialogue,ICD10
0,[doctor]: Good morning. How are you feeling to...,A047
1,"[doctor]: Good morning, Mrs. Doe. How are you ...",A047
2,"[doctor]: Hi Mr. Lee, how are you doing today?...",A047
3,"[doctor]: Good morning, how are you today?[pat...",A047
4,"[doctor]: Hi there, how are you doing today?[p...",A047
...,...,...
10232,[doctor]: Good morning! It's nice to see you a...,Z9981
10233,"```markdown[doctor] Hello, how are you doing t...",Z9981
10234,[doctor] Good morning. How are you feeling tod...,Z9981
10235,"[doctor] Hi Mr. Lee, how are you doing today?[...",Z9981


In [11]:
# Train test split

g = df.groupby('ICD10')

train_df = g.nth([1,2,3,4]).reset_index()
train_df = train_df.drop(["index"], axis=1)
val_df  = g.nth(0).reset_index()
val_df = val_df.drop(["index"], axis=1)

In [12]:
train_df

,Dialogue,ICD10
0,"[doctor]: Good morning, Mrs. Doe. How are you ...",A047
1,"[doctor]: Hi Mr. Lee, how are you doing today?...",A047
2,"[doctor]: Good morning, how are you today?[pat...",A047
3,"[doctor]: Hi there, how are you doing today?[p...",A047
4,"[doctor] Hi, how are you doing today?[patient]...",A0472
...,...,...
8131,"[doctor]: Hi there, I see you're here today fo...",Z992
8132,"```markdown[doctor] Hello, how are you doing t...",Z9981
8133,[doctor] Good morning. How are you feeling tod...,Z9981
8134,"[doctor] Hi Mr. Lee, how are you doing today?[...",Z9981


In [13]:
val_df

,Dialogue,ICD10
0,[doctor]: Good morning. How are you feeling to...,A047
1,"[doctor]: Hi Jane, I see you’re here today bec...",A0472
2,"[doctor]: Hi there, I see you're not feeling w...",A084
3,"[doctor]: Hi, how are you feeling today?[patie...",A09
4,[doctor]: Good morning. How are you feeling to...,A4101
...,...,...
2029,"[doctor]: Hello Maria, how are you feeling tod...",Z9889
2030,"[doctor]: Hello Maria, how are you feeling tod...",Z98890
2031,"[doctor]: Hello, how are you doing today?[pati...",Z9911
2032,"[doctor]: Good morning, Ms. Doe. How are you f...",Z992


In [14]:
# Export as CSV
# train_df.to_csv("training_data_embedding.csv")
# val_df.to_csv("validation_data_embedding.csv")

In [ ]:
# Export as JSON
train_df.to_json("train/training_finetuning_embedding.json", orient="records")
val_df.to_json("validation/validation_finetuning_embedding.json", orient="records")